In [51]:
END_INDEX=1

# replace with last file index: Ex. if last file is S (6).wav, set END_INDEX=6

In [52]:
# Prepare the environment
import io
import os
import subprocess
import time

from docx import Document
import IPython.display as ipd
from etils import epath as ep
from google.api_core.client_options import ClientOptions
from google.cloud.speech_v2 import SpeechClient
from google.cloud.speech_v2.types import cloud_speech
import jiwer
import pandas as pd
import plotly.graph_objs as go
from pydub import AudioSegment
import os 
from google.cloud import storage




PROJECT_ID = "verdant-branch-457906-a2" 
LOCATION = "us" # Using the multi-region "us" as it supports chirp_3
BUCKET_NAME = "bdav42"
BUCKET_URI = f"gs://{BUCKET_NAME}"

print(f"Using project {PROJECT_ID} in location {LOCATION} using bucket {BUCKET_URI}")



Using project verdant-branch-457906-a2 in location us using bucket gs://bdav42


In [54]:
# Setup common parameters for all requests
client = SpeechClient(
    client_options=ClientOptions(
        api_endpoint=f"{LOCATION}-speech.googleapis.com",
    )
)

speaker_diarization_config = cloud_speech.SpeakerDiarizationConfig(
        min_speaker_count=1,  # minimum number of speakers
        max_speaker_count=6,  # maximum expected number of speakers
    )

storage_client = storage.Client()
RECOGNIZER = client.recognizer_path(PROJECT_ID, LOCATION, "_")

RECOGNITION_CONFIG = cloud_speech.RecognitionConfig(
    auto_decoding_config=cloud_speech.AutoDetectDecodingConfig(),
    language_codes=["fa-IR"],
    model="chirp_3",
)
INPUT_LONG_AUDIO_SAMPLE_FILE_URI = ""

In [55]:
def parse_batch_recognize_response(response, audio_sample_file_uri):
    """
    Parses the batch recognize response and returns a list of transcripts.
    This function was missing in the original code.
    """
    transcriptions = []
    # Check if results exist for the file URI
    if audio_sample_file_uri in response.results:
        result_list = response.results[audio_sample_file_uri].transcript.results
        for result in result_list:
            transcript = result.alternatives[0].transcript
            transcriptions.append(transcript)
    return transcriptions

def transcribe_batch_chirp3(audio_uri: str, recognizer: str) -> cloud_speech.BatchRecognizeResults:
    """
    Function to perform batch transcription.
    The config is passed to this function to avoid duplication.
    """
    file_metadata = cloud_speech.BatchRecognizeFileMetadata(uri=audio_uri)

    request = cloud_speech.BatchRecognizeRequest(
        recognizer=recognizer,
        config=RECOGNITION_CONFIG,
        files=[file_metadata],
        recognition_output_config=cloud_speech.RecognitionOutputConfig(
            inline_response_config=cloud_speech.InlineOutputConfig(),
        ),
    )

    # Transcribes the audio into text
    operation = client.batch_recognize(request=request)

    print("Waiting for operation to complete...")
    response = operation.result(timeout=120)

    for result in response.results[audio_uri].transcript.results:
        print(f"Transcript: {result.alternatives[0].transcript}")
        print(f"Detected Language: {result.language_code}")
    
    return response.results[audio_uri].transcript


In [56]:
# Main Loop to process files
for i in range(1, END_INDEX + 1):
    FILE_NAME = f"S ({i}).wav"
    INPUT_LONG_AUDIO_SAMPLE_FILE_URI = (f"{BUCKET_URI}/Spch2txt/AudioInput/{FILE_NAME}")
    
    blob = storage_client.bucket(BUCKET_URI.replace("gs://", "")).blob(f"Spch2txt/AudioInput/{FILE_NAME}")
    if not blob.exists():
        print(f"Error: File '{INPUT_LONG_AUDIO_SAMPLE_FILE_URI}' does not exist in Google Cloud Storage.")
        exit(1)

    # Run the batch recognition operation
    operation = client.batch_recognize(
        request=cloud_speech.BatchRecognizeRequest(
            config=RECOGNITION_CONFIG,
            files=[cloud_speech.BatchRecognizeFileMetadata(uri=INPUT_LONG_AUDIO_SAMPLE_FILE_URI)],
            recognition_output_config=cloud_speech.RecognitionOutputConfig(
                inline_response_config=cloud_speech.InlineOutputConfig(),
            ),
            recognizer=RECOGNIZER,
        )
    )

    print(f"Waiting for transcribing {FILE_NAME} to complete...")
    response = operation.result()
    print(f"Operation completed for {FILE_NAME}.")

    batch_recognize_results = parse_batch_recognize_response(response, audio_sample_file_uri=INPUT_LONG_AUDIO_SAMPLE_FILE_URI)
    print(batch_recognize_results)
    
    # Save the results to a Word document
    document = Document()
    document.add_heading(f'Persian Translation Results {FILE_NAME}', level=0)

    for transcription in batch_recognize_results:
        document.add_paragraph(transcription)
        document.add_paragraph() # Add an empty paragraph for spacing   

    document.save(f'{FILE_NAME[:-4]}.docx')
    print(f"Persian translation saved to {FILE_NAME[:-4]}.docx")

print("🎉 All files processed!")

Waiting for transcribing S (1).wav to complete...
Operation completed for S (1).wav.
['سلام دوستان امیدوارم حالتون خوب باشه و تنتون در سلامت همونطور که مطلعید چند هفته پیش من دو متن فارسی و انگلیسی در مورد دیدگاه هفت سنگ و روش\u200cهای مداخله\u200cای برگرفته از اون رو خدمتتون تقدیم کردم با توجه به بازخورد\u200cهای دریافتی به نظر می\u200cرسه هر دو متن برای درک و تبین کافی مسئله یه کمی دشوار بودن به همین جهت و با توجه به اهمیت موضوع تصمیم گرفتم در تاریخ ۲۸ سپتامبر ۲۰۲۵ ساعت ۴ یکشنبه در این روز به وقت تورنتو یک موضوع بسیار کلیدی در سلامت روانی و رفتارمون یعنی داغ دیدگی و سوگ رو با دیدگاه هفت سنگ مورد بررسی قرار بدم تا هم به این موضوع مهم که تأثیرات بسیار عمیقی در زندگی ما داره و از طرف دیگه با کلیات و مفاهیم هفت سنگ که می\u200cتونه در خیلی از چیزای دیگه به ما کمک کنه به صورت نسبتاً مفصل به هر دوتاشون بپردازیم این جلسه در مجموع با اون ۲۰ دقیقه میانیش در مجموع ۳ ساعت خواهد بود و موضوع اون هم اینه که چقدر سوگ\u200cهای خود را می\u200cشناسیم و از اون\u200cها به سلامت گذشته\u200cایم همونطور که 